# EDA for the Taiwan dataset

This notebook does initial exploration of the Taiwan dataset to see what data is in there

**Author:** Ben Flint  
**Date:** 19/08/2026  
**Source:** https://www.semanticscholar.org/paper/The-comparisons-of-data-mining-techniques-for-the-Yeh-Lien/1cacac4f0ea9fdff3cd88c151c94115a9fddcf33, accessed: 26/08/2026  

# Background information on dataset

The dataset contains credit card payment data from October 2005, from a bank (a cash and credit card issuer) in Taiwan.

Among the total 25,000 observations, 5,529 observations (22.12%) are from cardholders with default payment.

The dataset contains a binary outcome variable — default payment (Yes = 1, No = 0) — and the following 23 explanatory variables:

- **X1** — Credit limit (NT dollar): includes both the individual consumer credit and their family (supplementary) credit
- **X2** — Gender (1 = male; 2 = female)
- **X3** — Education (1 = graduate school; 2 = university; 3 = high school; 4 = others)
- **X4** — Marital status (1 = married; 2 = single; 3 = others)
- **X5** — Age (years)

**X6–X11 — history of past payment (repayment status)**

Monthly repayment records tracked from April to September 2005:

| Variable | Month |
|---|---|
| X6 | September 2005 |
| X7 | August 2005 |
| X8 | July 2005 |
| X9 | June 2005 |
| X10 | May 2005 |
| X11 | April 2005 |

Measurement scale for repayment status:

| Value | Meaning |
|---|---|
| -1 | Pay duly (on time) |
| 1 | Payment delay for one month |
| 2 | Payment delay for two months |
| ... | ... |
| 8 | Payment delay for eight months |
| 9 | Payment delay for nine months and above |

**X12–X17 — amount of bill statement (NT dollar)**

| Variable | Month |
|---|---|
| X12 | September 2005 |
| X13 | August 2005 |
| X14 | July 2005 |
| X15 | June 2005 |
| X16 | May 2005 |
| X17 | April 2005 |

**X18–X23 — amount of previous payment (NT dollar)**

| Variable | Month |
|---|---|
| X18 | September 2005 |
| X19 | August 2005 |
| X20 | July 2005 |
| X21 | June 2005 |
| X22 | May 2005 |
| X23 | April 2005 |

In [16]:
import numpy as np
import pandas as pd
import sklearn as skl

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)



In [40]:
df = pd.read_excel("../data/raw/default of credit card clients.xls")
df.columns = [f"{col_name}_{row_1}" for (col_name, row_1) in zip(df.columns, df.iloc[0,:], strict=True)]
col_name_map = {
    'Unnamed: 0_ID': 'ID',
    'X1_LIMIT_BAL': 'X01_LIMIT_BAL',
    'X2_SEX': 'X02_SEX',
    'X3_EDUCATION': 'X03_EDUCATION',
    'X4_MARRIAGE': 'X04_MARRIAGE',
    'X5_AGE': 'X05_AGE',
    'X6_PAY_0': 'X06_PAY_0',
    'X7_PAY_2': 'X07_PAY_2',
    'X8_PAY_3': 'X08_PAY_3',
    'X9_PAY_4': 'X09_PAY_4'
    }
df = df.rename(columns=col_name_map)
df = df.drop(0, axis=0).reset_index(drop=True)
df = df.convert_dtypes()
df.head(10)

,ID,X01_LIMIT_BAL,X02_SEX,X03_EDUCATION,X04_MARRIAGE,X05_AGE,X06_PAY_0,X07_PAY_2,X08_PAY_3,X09_PAY_4,X10_PAY_5,X11_PAY_6,X12_BILL_AMT1,X13_BILL_AMT2,X14_BILL_AMT3,X15_BILL_AMT4,X16_BILL_AMT5,X17_BILL_AMT6,X18_PAY_AMT1,X19_PAY_AMT2,X20_PAY_AMT3,X21_PAY_AMT4,X22_PAY_AMT5,X23_PAY_AMT6,Y_default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0
5,6,50000,1,1,2,37,0,0,0,0,0,0,64400,57069,57608,19394,19619,20024,2500,1815,657,1000,1000,800,0
6,7,500000,1,1,2,29,0,0,0,0,0,0,367965,412023,445007,542653,483003,473944,55000,40000,38000,20239,13750,13770,0
7,8,100000,2,2,2,23,0,-1,-1,0,0,-1,11876,380,601,221,-159,567,380,601,0,581,1687,1542,0
8,9,140000,2,3,1,28,0,0,2,0,0,0,11285,14096,12108,12211,11793,3719,3329,0,432,1000,1000,1000,0
9,10,20000,1,3,2,35,-2,-2,-2,-2,-1,-1,0,0,0,0,13007,13912,0,0,0,13007,1122,0,0


In [41]:
df.columns

Index(['ID', 'X01_LIMIT_BAL', 'X02_SEX', 'X03_EDUCATION', 'X04_MARRIAGE',
       'X05_AGE', 'X06_PAY_0', 'X07_PAY_2', 'X08_PAY_3', 'X09_PAY_4',
       'X10_PAY_5', 'X11_PAY_6', 'X12_BILL_AMT1', 'X13_BILL_AMT2',
       'X14_BILL_AMT3', 'X15_BILL_AMT4', 'X16_BILL_AMT5', 'X17_BILL_AMT6',
       'X18_PAY_AMT1', 'X19_PAY_AMT2', 'X20_PAY_AMT3', 'X21_PAY_AMT4',
       'X22_PAY_AMT5', 'X23_PAY_AMT6', 'Y_default payment next month'],
      dtype='str')

In [42]:
dtypes = df.dtypes
dtypes

ID                              Int64
X01_LIMIT_BAL                   Int64
X02_SEX                         Int64
X03_EDUCATION                   Int64
X04_MARRIAGE                    Int64
X05_AGE                         Int64
X06_PAY_0                       Int64
X07_PAY_2                       Int64
X08_PAY_3                       Int64
X09_PAY_4                       Int64
X10_PAY_5                       Int64
X11_PAY_6                       Int64
X12_BILL_AMT1                   Int64
X13_BILL_AMT2                   Int64
X14_BILL_AMT3                   Int64
X15_BILL_AMT4                   Int64
X16_BILL_AMT5                   Int64
X17_BILL_AMT6                   Int64
X18_PAY_AMT1                    Int64
X19_PAY_AMT2                    Int64
X20_PAY_AMT3                    Int64
X21_PAY_AMT4                    Int64
X22_PAY_AMT5                    Int64
X23_PAY_AMT6                    Int64
Y_default payment next month    Int64
dtype: object

In [43]:
description_df = df.describe()

In [44]:
description_df = df.describe()
description_df = pd.pivot_table(description_df.reset_index(), columns='index')
description_df

index,25%,50%,75%,count,max,mean,min,std
ID,7500.75,15000.5,22500.25,30000.0,30000.0,15000.5,1.0,8660.398374
X01_LIMIT_BAL,50000.0,140000.0,240000.0,30000.0,1000000.0,167484.322667,10000.0,129747.661567
X02_SEX,1.0,2.0,2.0,30000.0,2.0,1.603733,1.0,0.489129
X03_EDUCATION,1.0,2.0,2.0,30000.0,6.0,1.853133,0.0,0.790349
X04_MARRIAGE,1.0,2.0,2.0,30000.0,3.0,1.551867,0.0,0.52197
X05_AGE,28.0,34.0,41.0,30000.0,79.0,35.4855,21.0,9.217904
X06_PAY_0,-1.0,0.0,0.0,30000.0,8.0,-0.0167,-2.0,1.123802
X07_PAY_2,-1.0,0.0,0.0,30000.0,8.0,-0.133767,-2.0,1.197186
X08_PAY_3,-1.0,0.0,0.0,30000.0,8.0,-0.1662,-2.0,1.196868
X09_PAY_4,-1.0,0.0,0.0,30000.0,8.0,-0.220667,-2.0,1.169139
